[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghiridharravi-athenatec/claude_code_flow/blob/main/run_in_colab.ipynb)

# Clinical Documentation Integrity Scorer -- Run in Colab (ngrok tunnel)

Clones the Clinical Documentation Integrity Scorer repo, installs backend + frontend dependencies, builds the frontend, starts one combined server (API + frontend, one port), and exposes it publicly via a single [ngrok](https://ngrok.com) tunnel so you can use the app from a normal browser tab. Along the way (Steps 5-6), it also runs the same record through two different Claude surfaces side by side -- a single Messages API call, and the Claude Agent SDK -- so you can see the difference in shape before you use the full app.

**Before running:** you'll need a free ngrok account and its authtoken (https://dashboard.ngrok.com/get-started/your-authtoken) -- you'll be prompted for it below. You'll also need your own Anthropic (Claude) API key -- entered once via a masked prompt in Step 4 for the demos below, and again in the running app's own browser UI once you reach it (Step 12); neither is ever written to disk or to this notebook.

Run the cells in order, top to bottom. This notebook assumes a Linux environment with Node.js/npm and git already available (true of the standard Colab runtime).

## 1. Configuration

In [ ]:
# TODO: replace with your repository's URL
REPO_URL = "https://github.com/ghiridharravi-athenatec/claude_code_flow.git"
BRANCH = "main"
REPO_DIR = "cdi-scorer-repo"

# One port, one process, one ngrok tunnel: the Flask backend serves both the
# API and the built frontend (see Step 10), so there's nothing to configure
# for a separate frontend port.
BACKEND_PORT = 5000

## 2. Clone the repository

In [ ]:
import os

if "<your-username>" in REPO_URL:
    raise ValueError("Set REPO_URL in the Configuration cell above to your actual repository URL before continuing.")

if os.path.isdir(REPO_DIR):
    print(f"'{REPO_DIR}' already exists -- skipping clone. Delete the folder and re-run this cell to re-clone.")
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

REPO_PATH = os.path.abspath(REPO_DIR)
BACKEND_DIR = os.path.join(REPO_PATH, "backend")
FRONTEND_DIR = os.path.join(REPO_PATH, "frontend")

print("Repo:    ", REPO_PATH)
print("Backend: ", BACKEND_DIR)
print("Frontend:", FRONTEND_DIR)

## 3. Install backend dependencies

In [ ]:
# libmagic is the native library python-magic wraps (SPEC.md Section 1.1) --
# Colab's Debian base doesn't ship it by default.
!apt-get -qq update && apt-get install -y -qq libmagic1

!pip install -q -r {BACKEND_DIR}/requirements.txt

print("Backend dependencies installed.")

Optional: pre-download the spaCy model used for PHI redaction on the backend's unhandled-error logging path (~400MB). Not required for the normal upload -> score flow -- skip unless you specifically want it warmed up in advance.

In [ ]:
# !python -m spacy download en_core_web_lg

## 4. Enter your Claude API key (for the demos below)

Steps 5 and 6 call Claude directly from this notebook, to demonstrate the two ways this application can score a record -- a single Messages API call (Part A, what the deployed app actually uses) and the Claude Agent SDK (Part B, a comparison). `getpass` hides what you type; the key lives only in this session's memory for the cells that need it, never written to disk or to this notebook.

This is separate from the deployed app itself: once you reach Step 12, the running app asks for your key again, in its own browser UI -- nothing here changes how the app handles keys (SPEC.md Section 6).

In [ ]:
import getpass

demo_api_key = getpass.getpass("Enter your Anthropic API key (used only for the Part A / Part B demos below): ")
print("Key stored in memory for this session only.")

## 5. Part A -- score a record with a single Messages API call

This is the real scoring path the deployed app uses -- imported and called directly, not reimplemented for this demo. `judge_record()` (in `backend/validation/llm_judge.py`) sends the record and the rubric to Claude in one Messages API call with a forced tool call, so the response is guaranteed structured JSON: a score for each of the ten dimensions. `score_record()` then runs the weighting, half-up rounding, decision-band lookup, and the two hard rules -- in plain Python, in `backend/validation/scorer.py`. That arithmetic is identical every run; only Claude's per-dimension judgment can vary.

The record scored below is the same synthetic fixture already used and disclosed throughout the Lab Guide (`backend/tests/golden/record_1.txt`) -- no real patient data, here or anywhere else in this notebook.

In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, BACKEND_DIR)

from config import Config
from validation.rubric_loader import load_rubric
from validation.scorer import score_record

rubric = load_rubric(Config.RUBRIC_PATH)

sample_record_path = Path(BACKEND_DIR) / "tests" / "golden" / "record_1.txt"
record_text = sample_record_path.read_text(encoding="utf-8")
print(f"Read {len(record_text)} characters from {sample_record_path.name} (synthetic fixture -- no real patient data)\n")

result = score_record(
    rubric,
    sample_record_path.name,
    record_text,
    datetime.now(timezone.utc).isoformat(),
    demo_api_key,
)

print("Per-dimension breakdown (Claude's judgment, then the plain-Python arithmetic):")
for d in result.dimension_results:
    print(f"  {d.rubric_id}  {d.name:<45} score={str(d.score):<4} weight={d.weight:<3} points={d.points_earned}")

print()
print(f"Overall: {result.overall.score_points} / {result.overall.score_max} -> {result.overall.decision}")
if result.overall.hard_rules_triggered:
    print("Hard rule(s) triggered:", [r.id for r in result.overall.hard_rules_triggered])

## 6. Part B -- hand the same record to the Claude Agent SDK

Same record, same rubric, a different surface -- and **not** part of the deployed app; this is a standalone comparison, confined to this notebook. Instead of one fully-specified call, the Claude Agent SDK gets a throwaway folder holding only the rubric and the record, and its own `Read`/`Write` tools -- it decides its own steps: read the rubric, read the record, work through all ten dimensions, and write a narrative review memo to that folder. Watch the `-> Using tool:` lines; that's the agent loop, made visible.

The Agent SDK drives the Claude Code CLI as a subprocess, so the next cell installs it first. Both are notebook-only additions -- nothing here is added to `backend/requirements.txt` or to the deployed app.

Unlike Part A, the agent's output is a freeform memo, not a number -- there's no arithmetic step afterward. That contrast is the point: reach for a single call when the task is fully specified (Part A), and for an agent when the steps aren't knowable in advance and you want it to produce the artifact itself.

In [ ]:
!pip install -q claude-agent-sdk
!npm install -g @anthropic-ai/claude-code
!claude --version

In [ ]:
import os
import shutil

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    ResultMessage,
    TextBlock,
    query,
)

agent_workspace = Path(REPO_PATH) / "agent_demo_workspace"
agent_workspace.mkdir(exist_ok=True)

# Only two files ever go into this throwaway folder: a copy of the real,
# read-only rubric, and the same synthetic fixture record used in Part A above
# -- no real patient data, ever.
shutil.copy(Config.RUBRIC_PATH, agent_workspace / "med_record_rubrics.json")
(agent_workspace / "record.txt").write_text(record_text, encoding="utf-8")

AGENT_SYSTEM_PROMPT = """You are a Clinical Documentation Integrity (CDI) reviewer working inside a folder.

Your job: read the medical record and the rubric, score the record against all
ten CDI dimensions, and write a documentation review memo.

Method:
1. Read med_record_rubrics.json to learn the weights, level descriptors, decision
   bands, and the two hard rules.
2. Read record.txt.
3. Score each dimension 1-5. Use N/E where the record gives no evidence at all --
   that is an absence of evidence, not a weakness, and must never be scored 1.
4. Where a dimension ties to a verifiable clinical criterion (e.g. KDIGO stage,
   GLIM/ASPEN criteria), compute it from the raw values documented in the record
   rather than trusting a stated diagnosis label at face value.
5. Write your memo to documentation_review_memo.md in the working folder.

Every score must quote the sentence from the record that supports it. If the
record is silent on a dimension, say so rather than inferring a value.
"""

agent_options = ClaudeAgentOptions(
    system_prompt=AGENT_SYSTEM_PROMPT,
    model=Config.CLAUDE_MODEL,
    cwd=str(agent_workspace),
    allowed_tools=["Read", "Write", "Glob", "Grep"],
    permission_mode="acceptEdits",
    max_turns=30,
)

agent_prompt = (
    "Score the medical record in record.txt against med_record_rubrics.json, "
    "then write your documentation review memo to documentation_review_memo.md."
)

os.environ["ANTHROPIC_API_KEY"] = demo_api_key
try:
    async def run_agent_demo():
        async for message in query(prompt=agent_prompt, options=agent_options):
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(block.text[:400])
                    else:
                        name = getattr(block, "name", type(block).__name__)
                        print(f"-> Using tool: {name}")
            elif isinstance(message, ResultMessage):
                print(f"[done] {message.result}")

    await run_agent_demo()
finally:
    os.environ.pop("ANTHROPIC_API_KEY", None)

### Read what the agent wrote

In [ ]:
from IPython.display import Markdown, display

memo_path = agent_workspace / "documentation_review_memo.md"
display(
    Markdown(memo_path.read_text(encoding="utf-8"))
    if memo_path.exists()
    else Markdown("_The agent did not write a memo. Re-run the cell above._")
)

## 7. Install frontend dependencies

In [ ]:
!npm install --prefix {FRONTEND_DIR}

## 8. Build the frontend for production

Built now, before anything is started or tunneled -- the build only needs to know that the API lives at a *relative* `/api/v1` path, not the eventual public URL, so there's no ordering dependency on the tunnel below.

In [ ]:
import os
import subprocess

frontend_build_env = os.environ.copy()
# Relative, not an absolute ngrok URL: once the API and the frontend are
# served from the same port (Step 10), the browser's own fetch calls to
# "/api/v1/..." resolve against whatever host is currently serving the page.
frontend_build_env["REACT_APP_API_BASE_URL"] = "/api/v1"

print("Building frontend (this can take a minute)...")
subprocess.run(
    ["npm", "run", "build"],
    cwd=FRONTEND_DIR,
    env=frontend_build_env,
    check=True,
)
print("Frontend build complete.")

## 9. Install and configure pyngrok

In [ ]:
!pip install -q pyngrok requests

import getpass
from pyngrok import ngrok

ngrok_authtoken = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(ngrok_authtoken)
del ngrok_authtoken  # never keep the token around longer than needed

print("ngrok configured.")

## 10. Start the combined app

Free ngrok accounts are limited to a single simultaneous tunnel/endpoint per agent session -- opening two tunnels (one for the backend API, one for a separate frontend static server, as earlier versions of this notebook did) either fails outright or hands back the same public URL for both, which is the "frontend and backend get the same ngrok URL" problem this version avoids entirely.

The fix is to serve both from the *same* process and port. This cell writes a small wrapper script -- kept in the cloned repo's working copy, not part of the actual project -- that loads the real, unmodified `create_app()` from `backend/app.py` and adds one extra route that serves the frontend's production build (from Step 8) as static files, falling back to `index.html` for the app's own page. `backend/app.py` itself is never changed -- running it directly, per Unit 7 of the lab guide, still works exactly as before.

In [ ]:
import os
import subprocess
import sys
import time

import requests

COMBINED_SERVER_PATH = os.path.join(REPO_PATH, "colab_combined_server.py")
FRONTEND_BUILD_DIR = os.path.join(FRONTEND_DIR, "build")

combined_server_source = f'''"""Colab-only glue: serves the real backend API (unmodified) plus the
frontend's production build from a single port, so this notebook needs only
one ngrok tunnel. Not part of the repository -- generated at runtime.
"""
import os
import sys

sys.path.insert(0, {BACKEND_DIR!r})
from app import create_app  # noqa: E402

from flask import send_from_directory

FRONTEND_BUILD_DIR = {FRONTEND_BUILD_DIR!r}

app = create_app()
# Point Flask's own /static/<path:filename> route (registered by Flask(__name__)
# in app.py, and otherwise unused since backend/static doesn't exist) at the
# frontend build's static/ subfolder -- Create React App serves its JS/CSS from
# exactly that /static/... URL shape, so this needs no extra route.
app.static_folder = os.path.join(FRONTEND_BUILD_DIR, "static")

@app.route("/", defaults={{"path": ""}})
@app.route("/<path:path>")
def serve_frontend(path):
    target = os.path.join(FRONTEND_BUILD_DIR, path)
    if path and os.path.isfile(target):
        return send_from_directory(FRONTEND_BUILD_DIR, path)
    return send_from_directory(FRONTEND_BUILD_DIR, "index.html")

if __name__ == "__main__":
    app.run(host="0.0.0.0", port={BACKEND_PORT!r})
'''

with open(COMBINED_SERVER_PATH, "w") as f:
    f.write(combined_server_source)


def wait_until_up(url, timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return True
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)
    return False


combined_log_path = os.path.join(REPO_PATH, "combined_server.log")
combined_log = open(combined_log_path, "w")

if "combined_process" in globals() and combined_process.poll() is None:
    print("Combined server already running (PID", combined_process.pid, ")")
else:
    combined_process = subprocess.Popen(
        [sys.executable, COMBINED_SERVER_PATH],
        cwd=REPO_PATH,
        stdout=combined_log,
        stderr=subprocess.STDOUT,
    )
    print("Started combined server (PID", combined_process.pid, ") -- logging to", combined_log_path)

if wait_until_up(f"http://localhost:{BACKEND_PORT}/api/v1/health"):
    print("App is up.")
else:
    print(f"App did not become healthy in time -- check {combined_log_path}")

## 11. Open the ngrok tunnel

Just one: the app (API + frontend, from Step 10) is already listening on `BACKEND_PORT`, so this opens a single tunnel to it.

In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError


def open_tunnel(port):
    for existing in ngrok.get_tunnels():
        if existing.config["addr"].endswith(f":{port}"):
            return existing
    return ngrok.connect(port, "http")


try:
    app_tunnel = open_tunnel(BACKEND_PORT)
except PyngrokNgrokError as exc:
    raise RuntimeError(
        "Could not open the ngrok tunnel. Check that your authtoken (Step 9) "
        "was entered correctly."
    ) from exc

app_public_url = app_tunnel.public_url
print("App tunnel:", app_public_url)

## 12. Published URL

In [ ]:
print("Open the app here:")
print(f"  {app_public_url}")
print()
print("The app will ask for your Claude API key in the browser -- it is never")
print("entered here, and is used only client-side plus for the single request")
print("it authorizes (SPEC.md Section 6).")

## 13. Shut down (optional)

Run this when you're done to stop the combined server and close the tunnel.

In [ ]:
if "combined_process" in globals() and combined_process.poll() is None:
    combined_process.terminate()
    print("Stopped combined_process (PID", combined_process.pid, ")")

ngrok.kill()
print("ngrok tunnel closed.")